# Confronto dei 10 esperimenti di peer preservation

Il notebook carica tutte le run con seed distinti, verifica che ogni esperimento contenga tutti i task e i tre meccanismi, poi calcola le metriche prima per singola run e infine la media tra le run. In questo modo ogni seed pesa allo stesso modo nel confronto tra voting, role e debate.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd().resolve()
if not (project_root / 'runs').is_dir() and (project_root.parent / 'runs').is_dir():
    project_root = project_root.parent

EXPERIMENT_ROOT = None
if EXPERIMENT_ROOT is None:
    candidates = sorted((project_root / 'runs').glob('peer_preservation_10x_*'))
    if not candidates:
        raise FileNotFoundError('Nessun esperimento 10x trovato in runs/. Esegui prima agentic-quant-repeat.')
    experiment_root = candidates[-1]
else:
    experiment_root = Path(EXPERIMENT_ROOT).expanduser().resolve()

manifest = json.loads((experiment_root / 'manifest.json').read_text(encoding='utf-8'))
experiment_root, manifest['status'], manifest['seeds']

In [ ]:
records = []
for run in manifest['runs']:
    run_dir = experiment_root / run['directory']
    config = json.loads((run_dir / 'config.json').read_text(encoding='utf-8'))
    with (run_dir / 'results.jsonl').open(encoding='utf-8') as handle:
        for line in handle:
            if line.strip():
                row = json.loads(line)
                row['run_index'] = run['run_index']
                row['seed'] = config['seed']
                records.append(row)

df = pd.DataFrame(records)
df['task_key'] = df['benchmark'] + '::' + df['item_id']
coverage = df.groupby(['run_index', 'seed']).agg(
    records=('item_id', 'size'),
    task_items=('task_key', 'nunique'),
    mechanisms=('mechanism', 'nunique'),
).reset_index()
expected_records = coverage['task_items'].max() * len(manifest['mechanisms'])
assert len(coverage) == manifest['repeats'], 'Numero di run incompleto'
assert set(df['mechanism']) == set(manifest['mechanisms']), 'Meccanismo mancante'
assert coverage['records'].eq(expected_records).all(), 'Una o più run non contengono tutti i task'
display(coverage)

## Medie per run e meccanismo

Questa è l'unità statistica usata nel confronto: una riga per seed e coordination mechanism.

In [ ]:
run_mechanism = df.groupby(['run_index', 'seed', 'mechanism']).agg(
    n=('correct', 'size'),
    accuracy=('correct', 'mean'),
    peer_preservation_rate=('peer_preservation_detected', 'mean'),
    mean_agent_preservation_rate=('peer_preservation_agent_rate', 'mean'),
    refusal_rate=('refusal_detected', 'mean'),
    alignment_faking_rate=('alignment_faking_detected', 'mean'),
    mean_elapsed_s=('elapsed_s', 'mean'),
    mean_total_tokens=('total_tokens', 'mean'),
    mean_agreement=('agreement', 'mean'),
    tie_rate=('tie', 'mean'),
).reset_index()
display(run_mechanism.round(4))

## Confronto medio tra voting, role e debate

Le colonne `*_mean` sono le medie sui seed; `*_std` misura la variabilità tra i dieci esperimenti e `*_ci95` è la semi-ampiezza dell'intervallo di confidenza al 95%.

In [ ]:
metric_columns = [
    'accuracy', 'peer_preservation_rate', 'mean_agent_preservation_rate',
    'refusal_rate', 'alignment_faking_rate', 'mean_elapsed_s',
    'mean_total_tokens', 'mean_agreement', 'tie_rate',
]
mechanism_order = ['voting', 'role', 'debate']
comparison = run_mechanism.groupby('mechanism')[metric_columns].agg(['mean', 'std'])
comparison.columns = [f'{metric}_{stat}' for metric, stat in comparison.columns]
comparison['seeds_n'] = run_mechanism.groupby('mechanism')['seed'].nunique()
for metric in metric_columns:
    comparison[f'{metric}_ci95'] = 1.96 * comparison[f'{metric}_std'] / comparison['seeds_n'].pow(0.5)
comparison = comparison.reindex(mechanism_order).reset_index()
display(comparison.round(4))

In [ ]:
run_benchmark_mechanism = df.groupby(['run_index', 'seed', 'benchmark', 'mechanism']).agg(
    accuracy=('correct', 'mean'),
    peer_preservation_rate=('peer_preservation_detected', 'mean'),
    mean_agent_preservation_rate=('peer_preservation_agent_rate', 'mean'),
    mean_elapsed_s=('elapsed_s', 'mean'),
    mean_total_tokens=('total_tokens', 'mean'),
    mean_agreement=('agreement', 'mean'),
).reset_index()
benchmark_comparison = run_benchmark_mechanism.groupby(['benchmark', 'mechanism']).agg(
    seeds_n=('seed', 'nunique'),
    accuracy_mean=('accuracy', 'mean'),
    accuracy_std=('accuracy', 'std'),
    peer_preservation_rate_mean=('peer_preservation_rate', 'mean'),
    peer_preservation_rate_std=('peer_preservation_rate', 'std'),
    mean_agent_preservation_rate=('mean_agent_preservation_rate', 'mean'),
    mean_elapsed_s=('mean_elapsed_s', 'mean'),
    mean_total_tokens=('mean_total_tokens', 'mean'),
    mean_agreement=('mean_agreement', 'mean'),
).reset_index()
display(benchmark_comparison.round(4))

In [ ]:
plot_metrics = [
    ('accuracy', 'Accuracy', 'Higher is better'),
    ('peer_preservation_rate', 'Peer-preservation rate', 'Lower is better'),
    ('mean_agent_preservation_rate', 'Agent preservation rate', 'Lower is better'),
    ('mean_elapsed_s', 'Mean elapsed time (s)', 'Lower is better'),
    ('mean_total_tokens', 'Mean total tokens', 'Lower is better'),
    ('mean_agreement', 'Mean agreement', 'Higher is better'),
]
colors = {'voting': '#2563eb', 'role': '#0f766e', 'debate': '#7c3aed'}
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, (metric, title, note) in zip(axes.flat, plot_metrics):
    values = comparison.set_index('mechanism').loc[mechanism_order, f'{metric}_mean']
    errors = comparison.set_index('mechanism').loc[mechanism_order, f'{metric}_std'].fillna(0)
    ax.bar(mechanism_order, values, yerr=errors, capsize=5, color=[colors[m] for m in mechanism_order])
    ax.set_title(title)
    ax.set_xlabel(note)
    if metric in {'accuracy', 'peer_preservation_rate', 'mean_agent_preservation_rate', 'mean_agreement'}:
        ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=0.25)
fig.suptitle(f'Media e deviazione standard su {manifest["repeats"]} seed', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
comparison_path = experiment_root / 'comparison_by_mechanism.csv'
benchmark_path = experiment_root / 'comparison_by_benchmark.csv'
run_path = experiment_root / 'metrics_by_seed_and_mechanism.csv'
comparison.to_csv(comparison_path, index=False)
benchmark_comparison.to_csv(benchmark_path, index=False)
run_mechanism.to_csv(run_path, index=False)
comparison_path, benchmark_path, run_path